<a href="https://colab.research.google.com/github/Renata-Nascimento/Projeto-U02-CD/blob/main/notebooks/02_limpeza_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, re, sys, glob, unicodedata
import numpy as np
import pandas as pd
import requests

# Carregando os dados do GitHub
base = pd.read_csv('https://raw.githubusercontent.com/Renata-Nascimento/Projeto-U02-CD/main/saida/base_municipios_2020_2023.csv')

In [2]:
# Extraindo as colunas referentes a municípios
municipios = base[['cod_ibge', 'municipio', 'mesorregiao', 'regiao_intermediaria']].drop_duplicates().reset_index(drop=True)

print(len(municipios), 'municípios')
municipios.head()

167 municípios


,cod_ibge,municipio,mesorregiao,regiao_intermediaria
0,2400109,Acari,Central Potiguar,Caicó
1,2400208,Assú,Oeste Potiguar,Mossoró
2,2400307,Afonso Bezerra,Central Potiguar,Mossoró
3,2400406,Água Nova,Oeste Potiguar,Mossoró
4,2400505,Alexandria,Oeste Potiguar,Mossoró


In [3]:
# Exibir informações da base consolidada
print(f"Tamanho final da base unificada: {base.shape[0]} linhas e {base.shape[1]} colunas.\n")
display(base.head())

Tamanho final da base unificada: 668 linhas e 21 colunas.



,cod_ibge,municipio,mesorregiao,regiao_intermediaria,ano,rep_icms,rep_ipi,rep_ipva,rep_royalties,rep_total,...,pib_mil_reais,fator_ipca,cad_baixa_renda,cad_pessoas,rep_total_real,pib_real_mil,rep_per_capita,pib_per_capita,pct_baixa_renda,porte
0,2400109,Acari,Central Potiguar,Caicó,2020,3.116225e+06,3279.713210,385048.780,44590.552676,3.549144e+06,...,132249.0,1.237887,5656,6675,4.393441e+06,1.637094e+05,414.592918,15448.653756,53.373596,10–20 mil
1,2400109,Acari,Central Potiguar,Caicó,2021,3.648454e+06,3571.667577,422640.475,77605.250704,4.152271e+06,...,141992.0,1.143000,5851,6940,4.746044e+06,1.622968e+05,447.866760,15315.352846,55.213740,10–20 mil
2,2400109,Acari,Central Potiguar,Caicó,2022,3.979198e+06,4876.969709,491858.065,113601.781011,4.589534e+06,...,173700.0,1.045936,6052,7262,4.800358e+06,1.816790e+05,452.992135,17144.382242,57.110503,10–20 mil
3,2400109,Acari,Central Potiguar,Caicó,2023,4.872919e+06,9948.103018,586816.705,96841.441298,5.566526e+06,...,195828.0,1.000000,5048,7041,5.566526e+06,1.958280e+05,525.292590,18479.569690,47.636123,10–20 mil
4,2400208,Assú,Oeste Potiguar,Mossoró,2020,1.939472e+07,20412.232329,2109941.855,277522.046181,2.180259e+07,...,1169390.0,1.237887,30759,34268,2.698916e+07,1.447573e+06,477.718030,25622.579695,54.444562,mais de 50 mil


### Análise de Qualidade dos Dados (`df_analise`)

Vamos executar um diagnóstico completo para garantir a integridade dos nossos dados antes de iniciar análises mais profundas.

In [4]:
print("=== 1. VERIFICAÇÃO DE VALORES NULOS ===")
missing_data = base.isnull().sum()
display(missing_data[missing_data > 0] if (missing_data > 0).any() else "Nenhum valor nulo encontrado!")

print("\n=== 2. RESUMO ESTATÍSTICO DOS VALORES FINANCEIROS ===")
display(base[['rep_total_real']].describe().apply(lambda s: s.apply('{0:.2f}'.format)))

print("\n=== 3. ESTRUTURA E TIPOS DE COLUNA ===")
display(base.dtypes)

=== 1. VERIFICAÇÃO DE VALORES NULOS ===


'Nenhum valor nulo encontrado!'


=== 2. RESUMO ESTATÍSTICO DOS VALORES FINANCEIROS ===


,rep_total_real
count,668.00
mean,13237120.52
std,45456722.92
min,1816676.35
25%,2618684.94
50%,3575888.38
75%,8430024.10
max,514431832.73



=== 3. ESTRUTURA E TIPOS DE COLUNA ===


,0
cod_ibge,int64
municipio,object
mesorregiao,object
regiao_intermediaria,object
ano,int64
rep_icms,float64
rep_ipi,float64
rep_ipva,float64
rep_royalties,float64
rep_total,float64


### Formatando a Visualização de Valores Monetários

Vamos usar o styler do pandas para aplicar a formatação de moeda brasileira (`R$`) nas colunas financeiras.

In [5]:
# Criando a nova coluna expressando o PIB diretamente em milhões de reais
base['pib_real_milhoes'] = base['pib_real_mil'] / 1000.0
base.drop('pib_real_mil', axis=1, inplace=True)

base['pib_milhoes_reais'] = base['pib_mil_reais'] / 1000.0
base.drop('pib_mil_reais', axis=1, inplace=True)

In [8]:
# Criando um mapeamento de formatação para as colunas
# Converter permanentemente as colunas 'cod_ibge' e 'ano' para string (texto) para evitar formatação numérica
base['cod_ibge'] = base['cod_ibge'].astype(str)
base['ano'] = base['ano'].astype(str)

formatacao = {
    'pib_real_milhoes': 'R$ {:,.2f} milhões',
    'pib_milhoes_reais': 'R$ {:,.2f} milhões',
    'rep_per_capita': 'R$ {:,.2f}',
    'pib_per_capita': 'R$ {:,.2f}',
    'rep_total_real': 'R$ {:,.2f}',
    'rep_total': 'R${:,.2f}',
    'rep_icms': 'R${:,.2f}',
    'rep_ipi': 'R${:,.2f}',
    'rep_ipva': 'R${:,.2f}',
    'rep_royalties': 'R${:,.2f}',
    'pct_baixa_renda': '{:.2f}%'
}

# Exibindo as primeiras linhas do DataFrame com a formatação aplicada
base.head().style.format(formatacao, thousands='.', decimal=',')

,cod_ibge,municipio,mesorregiao,regiao_intermediaria,ano,rep_icms,rep_ipi,rep_ipva,rep_royalties,rep_total,populacao,fator_ipca,cad_baixa_renda,cad_pessoas,rep_total_real,rep_per_capita,pib_per_capita,pct_baixa_renda,porte,pib_real_milhoes,pib_milhoes_reais
0,2400109,Acari,Central Potiguar,Caicó,2020,"R$3.116.225,09","R$3.279,71","R$385.048,78","R$44.590,55","R$3.549.144,13",10.597,"1,237887",5.656,6.675,"R$ 4.393.441,16","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,"R$ 163,71 milhões","R$ 132,25 milhões"
1,2400109,Acari,Central Potiguar,Caicó,2021,"R$3.648.453,70","R$3.571,67","R$422.640,47","R$77.605,25","R$4.152.271,09",10.597,"1,143000",5.851,6.940,"R$ 4.746.044,05","R$ 447,87","R$ 15.315,35","55,21%",10–20 mil,"R$ 162,30 milhões","R$ 141,99 milhões"
2,2400109,Acari,Central Potiguar,Caicó,2022,"R$3.979.197,68","R$4.876,97","R$491.858,07","R$113.601,78","R$4.589.534,50",10.597,"1,045936",6.052,7.262,"R$ 4.800.357,65","R$ 452,99","R$ 17.144,38","57,11%",10–20 mil,"R$ 181,68 milhões","R$ 173,70 milhões"
3,2400109,Acari,Central Potiguar,Caicó,2023,"R$4.872.919,32","R$9.948,10","R$586.816,70","R$96.841,44","R$5.566.525,57",10.597,"1,000000",5.048,7.041,"R$ 5.566.525,57","R$ 525,29","R$ 18.479,57","47,64%",10–20 mil,"R$ 195,83 milhões","R$ 195,83 milhões"
4,2400208,Assú,Oeste Potiguar,Mossoró,2020,"R$19.394.717,29","R$20.412,23","R$2.109.941,86","R$277.522,05","R$21.802.593,42",56.496,"1,237887",30.759,34.268,"R$ 26.989.157,84","R$ 477,72","R$ 25.622,58","54,44%",mais de 50 mil,"R$ 1.447,57 milhões","R$ 1.169,39 milhões"
